# C15 — Identification du locuteur sur OpenSLR 57 (P11, E1)

Ce notebook mesure si une voix est reconnue, avec deux encodeurs gelés : **ECAPA-TDNN**
(SpeechBrain, fait pour le locuteur) et **mHuBERT** (le même encodeur que pour les émotions).

Protocole (P11) : pour chaque locuteur qui a au moins 10 phrases, 5 phrases tirées au hasard
(graine 42) « enregistrent » sa voix, les autres servent au test. Les locuteurs sont répartis
en deux groupes : la **calibration** choisit le seuil « inconnu » (et la couche de mHuBERT),
l'**évaluation** donne les scores. Le découpage est versionné dans
`data/splits/openslr57_speakers.csv`.

Il faut **Internet activé** et un **GPU** (Accelerator : GPU T4 x2 ou P100). Le notebook
n'écrit qu'un fichier de résultats, `c15_speaker_openslr57.json`, sans audio (I8).

## Démarrage

On clone le dépôt (branche `BRANCH`), on l'installe en mode éditable (I9), puis on installe
SpeechBrain, qui n'est pas présent d'office sur Kaggle, sans toucher au PyTorch de Kaggle.

In [ ]:
import sys

BRANCH = "feat/c15-evaluation-locuteur"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}
# SpeechBrain sans ses dépendances : sinon pip remplace le PyTorch de Kaggle, qui ne se charge
# plus. Ses autres dépendances (numpy, scipy, huggingface_hub…) sont déjà sur Kaggle.
%pip install -q --no-deps speechbrain hyperpyyaml
# hyperpyyaml (lecture des fichiers de SpeechBrain) ne suit pas les versions récentes de
# ruamel.yaml : on fixe une version compatible.
%pip install -q "ruamel.yaml>=0.17.28,<0.18"

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## a. GPU

Sans GPU, mHuBERT prendrait plusieurs heures sur les 15 000 phrases : on s'arrête tout de suite.
On rétablit aussi une fonction de torchaudio dont SpeechBrain a encore besoin.

In [ ]:
import torch
import torchaudio


def list_audio_backends() -> list[str]:
    # Remplace une fonction retirée des versions récentes de torchaudio, que SpeechBrain appelle
    # encore. On ne lit jamais de fichier par torchaudio (load_audio s'en charge, D1).
    return ["soundfile"]


if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = list_audio_backends
print("torch", torch.__version__, "· torchaudio", torchaudio.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("Pas de GPU : choisir Accelerator → GPU dans Session options.")
device = torch.device("cuda")
print(torch.cuda.get_device_name(0))

## b. Téléchargement d'OpenSLR 57

Archive officielle (environ 1,8 Go), supprimée dès qu'elle est décompressée.

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

from echolens.core.data.manifest import dataset_root

DOWNLOAD_DIR = Path("/kaggle/working/downloads")
OPENSLR57_TAR_URL = "https://openslr.elda.org/resources/57/African_Accented_French.tar.gz"

archive_path = DOWNLOAD_DIR / "openslr57.tar.gz"
extract_dir = DOWNLOAD_DIR / "openslr57"
archive_path.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(OPENSLR57_TAR_URL, archive_path)
with tarfile.open(archive_path) as archive:
    archive.extractall(extract_dir, filter="data")
archive_path.unlink()
openslr57_root = sorted(extract_dir.glob("**/African_Accented_French"))[0]

# Convention <datasets_dir>/<id> des manifestes (docs/datasets.md).
datasets_dir = Path("/kaggle/working/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
link = dataset_root("openslr57", datasets_dir)
if link.is_symlink():
    link.unlink()
link.symlink_to(openslr57_root)
print(link, "→", openslr57_root)

## c. Découpage P11

On refait le découpage avec le code du dépôt et on vérifie qu'il est identique au fichier
versionné : le tirage est déterministe (graine 42).

In [ ]:
from collections import Counter

from echolens.core.data.manifest import load_manifest
from echolens.core.data.speaker_enrollment import (
    load_speaker_config,
    save_enrollment,
    split_enrollment,
)

config = load_speaker_config(Path(CLONE_DIR) / "configs/speaker.yaml")
rows = load_manifest(Path(CLONE_DIR) / "data/manifests/openslr57.csv")
speakers = split_enrollment(rows, config)

check_path = Path("/kaggle/working/check_split.csv")
save_enrollment(speakers, check_path)
versioned_path = Path(CLONE_DIR) / "data/splits/openslr57_speakers.csv"
same_split = check_path.read_text(encoding="utf-8") == versioned_path.read_text(encoding="utf-8")
check_path.unlink()
if not same_split:
    raise RuntimeError("Le découpage diffère du fichier versionné : ne pas continuer.")

# Seules les phrases des locuteurs gardés passent dans les encodeurs.
kept_paths = set()
for speaker in speakers:
    kept_paths.update(speaker.enrollment_paths)
    kept_paths.update(speaker.test_paths)
kept_rows = [row for row in rows if row.path in kept_paths]
print("Locuteurs gardés :", len(speakers))
print("Par sous-corpus et groupe :", Counter((s.subset, s.group) for s in speakers))
print("Phrases à encoder :", len(kept_rows))

## d. ECAPA-TDNN

Un vecteur de 192 valeurs par phrase, avec la révision figée dans `configs/speaker.yaml`.

In [ ]:
import time

from echolens.training.speaker_encoders import embed_rows, load_ecapa
from echolens.training.speaker_eval import evaluate_speakers

encoder_configs = {}
for encoder_config in config.encoders:
    encoder_configs[encoder_config.id] = encoder_config

ecapa_config = encoder_configs["ecapa-voxceleb"]
ecapa_revision = ecapa_config.revision
print("Révision ECAPA :", ecapa_revision)

ecapa_model = load_ecapa(ecapa_config, Path("/kaggle/working/ecapa"), device)
start = time.perf_counter()
ecapa_embeddings = embed_rows(kept_rows, datasets_dir, ecapa_model, ecapa_config, config, device)
ecapa_seconds = time.perf_counter() - start
ecapa_result = evaluate_speakers(speakers, ecapa_embeddings)
print(ecapa_result)

## e. mHuBERT

Une moyenne par couche (13 × 768). La couche gardée est celle qui a la plus faible EER sur le
groupe de **calibration** ; le groupe d'évaluation n'intervient pas dans ce choix.

In [ ]:
from echolens.training.speaker_encoders import load_ssl
from echolens.training.speaker_eval import (
    best_layer,
    group_scores,
    select_layer,
    speakers_of_group,
    verification_eer,
)

mhubert_config = encoder_configs["mhubert-147"]
mhubert_encoder = load_ssl(mhubert_config, config.sample_rate, device)
start = time.perf_counter()
mhubert_layers = embed_rows(
    kept_rows, datasets_dir, mhubert_encoder, mhubert_config, config, device
)
mhubert_seconds = time.perf_counter() - start

# EER de calibration couche par couche, affichée pour le rapport.
calibration = speakers_of_group(speakers, "calibration")
calibration_eer_by_layer = []
layer_count = next(iter(mhubert_layers.values())).shape[0]
for layer in range(layer_count):
    layer_scores = group_scores(calibration, select_layer(mhubert_layers, layer))
    layer_eer, _ = verification_eer(layer_scores)
    calibration_eer_by_layer.append(layer_eer)
    print(f"Couche {layer} : EER de calibration {layer_eer:.4f}")

mhubert_layer = best_layer(speakers, mhubert_layers)
mhubert_result = evaluate_speakers(speakers, select_layer(mhubert_layers, mhubert_layer))
print("Couche gardée :", mhubert_layer)
print(mhubert_result)

## f. Résultats

Un seul fichier JSON, à ajouter au dépôt dans `training/results/`.

In [ ]:
import json
from dataclasses import asdict

results = {
    "protocol": "P11",
    "dataset": "openslr57",
    "seed": config.seed,
    "enrollment_per_speaker": config.enrollment_per_speaker,
    "min_utterances_per_speaker": config.min_utterances_per_speaker,
    "max_seconds": config.max_seconds,
    "speakers_kept": len(speakers),
    "gpu": torch.cuda.get_device_name(0),
    "encoders": {
        "ecapa-voxceleb": {
            "revision": ecapa_revision,
            "compute_seconds": round(ecapa_seconds, 1),
            **asdict(ecapa_result),
        },
        "mhubert-147": {
            "revision": mhubert_config.revision,
            "layer": mhubert_layer,
            "calibration_eer_by_layer": calibration_eer_by_layer,
            "compute_seconds": round(mhubert_seconds, 1),
            **asdict(mhubert_result),
        },
    },
}
result_path = Path("/kaggle/working/c15_speaker_openslr57.json")
result_path.write_text(json.dumps(results, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(results, indent=2, ensure_ascii=False))

## Nettoyage

On supprime le clone, les audios et les poids : les sorties ne gardent que le JSON (I8).

In [ ]:
import shutil

shutil.rmtree(CLONE_DIR, ignore_errors=True)
shutil.rmtree(datasets_dir, ignore_errors=True)
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
shutil.rmtree("/kaggle/working/ecapa", ignore_errors=True)
remaining = [path.name for path in Path("/kaggle/working").iterdir()]
print(sorted(remaining))